<a href="https://colab.research.google.com/github/francescodeciano/CV-project2/blob/main/CV2026_project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
!apt-get install -y aria2 > /dev/null

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import os
import glob
import random
import shutil
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path

# Globals

In [ ]:
#IF YOU ARE TESTING A DIFFERENT VALUE OF ALPHA EXECUTE AGAIN THIS CELL
#Set seed for reproducibility for every library, also for gpu
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
  torch.cuda.manual_seed(SEED)
  torch.cuda.manual_seed_all(SEED)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

#Set the colab device to gpu, if not available to cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#Hyperparameters
LEARNING_RATE = 1e-3
EPOCHS = 20
BATCH_SIZE = 64
ALPHA = 0.5             #choose among {0.0, 0.5, 1.0}
PATIENCE = 3            #for early stopping

#Creation of directories
DATA_DIR = "data"
MODELS_DIR = "models"
PLOTS_DIR = "plots"
METRICS_DIR = "metrics"
CHECKPOINT_DIR = "checkpoints"
for folder in [DATA_DIR, MODELS_DIR, PLOTS_DIR, METRICS_DIR, CHECKPOINT_DIR]:
  os.makedirs(folder, exist_ok=True)

# Utils

In [ ]:
def class_balance(full_ds, train_split, val_split, test_split):
  """Verifies class balance in the training, validation and test subsets
  before the training process"""
  rf_names = {0: "Real", 1: "Fake"}
  trans_names = {0: "Original", 1: "Internet", 2: "Re-digitized"}

  splits = {
      "Train": train_split.indices,
      "Validation": val_split.indices,
      "Test": test_split.indices
  }

  records = []
  for split, indices in splits.items():
    for i in indices:
      rf_label = full_ds.labels_rf[i]
      trans_label = full_ds.labels_trans[i]
      combination = f"{rf_names[rf_label]} - {trans_names[trans_label]}"
      records.append({
          "Split": split,
          "Combination": combination
      })
  df_balance = pd.DataFrame(records)

  for split in splits.keys():
    print(f"\n{split} Set Combinations")
    counts = df_balance[df_balance["Split"] == split]["Combination"].value_counts().to_dict()
    for combo, count in counts.items():
      print(f"  {combo}: {count}")

  costum_palette = {
      "Fake - Internet": "#ee6055",
      "Real - Internet": "#60d394",
      "Fake - Original": "#aaf683",
      "Real - Original": "#ffd97d",
      "Fake - Re-digitized": "#ff9b85",
      "Real - Re-digitized": "#D85A7F"
  }
  sns.set_theme(style="whitegrid")
  plt.figure(figsize=(15,6))

  ax = sns.countplot(
      data=df_balance,
      x="Split",
      hue="Combination",
      palette=costum_palette,
      linewidth=0,
      width=0.85,
      gap=0
  )

  plt.title("Distribution of the 6 Class Combinations per Split", fontsize=14, fontweight="bold", pad=15)
  plt.ylabel("Samples", fontsize=12)
  plt.xlabel("Dataset split", fontsize=12)
  plt.legend(title="Combinations", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=11, title_fontsize=12)

  plt.legend(
      title="Combinations",
      loc="upper right",
      fontsize=9.5,
      title_fontsize=10.5,
      framealpha=0.9
  )

  for container in ax.containers:
    ax.bar_label(container, fontsize=10, fontweight="bold", padding=3)

  plt.tight_layout()
  print("\n")
  plt.show()

In [ ]:
#Early stopping to avoid overfitting
class EarlyStopping:
  """When the training loss decreases but the validation loss
  does not, it means that the model is adapting too much to the
  training set, leading to overfitting. When this happens, with
  EarlySTopping the training is interrupted to avoid overfitting"""
  #Patience = number of epochs without improvements counted by self.counter
  #Delta = minimum improvement
  def __init__(self, patience=2, verbose=False, delta=0.0):
    self.patience = patience
    self.verbose = verbose
    self.delta = delta
    self.counter = 0
    self.best_score = None
    self.early_stop = False

  def __call__(self, val_loss):
    if self.best_score is None:
      self.best_score = val_loss
    elif val_loss > self.best_score - self.delta:
      self.counter += 1
      if self.verbose:
        print(f"EarlyStopping counter: {self.counter}/{self.patience}")
      if self.counter >= self.patience:
        self.early_stop = True
    else:
      self.best_score = val_loss
      self.counter = 0

In [ ]:
def save_checkpoint(model, optimizer, epoch, loss, checkpoint_dir=CHECKPOINT_DIR):
  """Save model checkpoint"""
  os.makedirs(checkpoint_dir, exist_ok=True)
  checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pt")
  torch.save({
      "epoch": epoch,
      "model_state_dict": model.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
      "loss": loss,
  }, checkpoint_path)

  return checkpoint_path

In [ ]:
def save_best_model(model, optimizer, epoch, loss, best_dir=MODELS_DIR):
  """Save the best model in a separate directory"""
  best_path = os.path.join(best_dir, f"final_model_alpha{ALPHA}.pt")
  torch.save({
      "epoch": epoch,
      "model_state_dict": model.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
      "loss": loss,
  }, best_path)
  print(f"New best model saved in: {best_path}")

In [ ]:
def clean_checkpoints(checkpoint_dir=CHECKPOINT_DIR):
  """Eliminate the entire checkpoint folder after finding the best model at the
  end of the whole training process"""
  if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
    print("All the intermediate checkpoints have been deleted to save memory")

In [ ]:
def metrics(df, rf_mapping, trans_mapping):
  """df contains the results, the other two arguments are mapping between
  numbers and words used in the dataset"""
  #Real/fake section
  print("\nReal/Fake Classification")
  print(classification_report(df["true_rf"], df["predicted_rf"], target_names=rf_mapping.values()))

  print("\nTransformation Classification")
  print(classification_report(df["true_trans"], df["predicted_trans"], target_names=trans_mapping.values()))

  #fill the table with the obtained values
  df["is_correct_rf"] = df["true_rf"] == df["predicted_rf"]
  df["Trans_label"] = df["true_trans"].map(trans_mapping)
  df["RF_label"] = df["true_rf"].map(rf_mapping)

  #Transformation section
  print("\nReal/Fake Accuracy per Transformation")
  #creare sub-groups(real and trasmitted, etc...) computes the mean and then
  #make them readable in a grid structure
  breakdown = df.groupby(["Trans_label", "RF_label"])["is_correct_rf"].mean().unstack()
  display(breakdown)

  #convert to csv
  breakdown.to_csv(os.path.join(METRICS_DIR, f"breakdown_alpha{ALPHA}.csv"))

  return breakdown

In [ ]:
def plot_test_results(df, rf_mapping, trans_mapping):
  """df contains the results, the other two arguments are mapping between
  numbers and words used in the dataset.
  Creates the confusion matrices for the two tasks"""
  plt.figure(figsize=(15,6))

  #Real/Fake matrix
  plt.subplot(1, 2, 1)
  cm_rf = confusion_matrix(df["true_rf"], df["predicted_rf"])
  sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues",
              xticklabels=rf_mapping.values(), yticklabels=rf_mapping.values())
  plt.title("Confusion matrix: Real/Fake")

  #Transformation matrix
  plt.subplot(1, 2, 2)
  cm_trans = confusion_matrix(df["true_trans"], df["predicted_trans"])
  sns.heatmap(cm_trans, annot=True, fmt="d", cmap="Greens",
              xticklabels=trans_mapping.values(), yticklabels=trans_mapping.values())
  plt.title("Confusion matrix: Transformation")

  plt.tight_layout()

  plt.savefig(os.path.join(PLOTS_DIR, f"plot_alpha{ALPHA}.png"), dpi=300)

  plt.show()

In [ ]:
#plot of the ablation study only if all three values are tested and the model is saved
def ablation_plot():
  alphas = [0.0, 0.5, 1.0]
  rf_accuracies = []
  trans_accuracies = []

  for alpha in alphas:
    model_path = os.path.join(MODELS_DIR, f"final_model_alpha{alpha}.pt")
    if os.path.exists(model_path):
      checkpoint = torch.load(model_path, map_location=device)
      model.load_state_dict(checkpoint["model_state_dict"])
      model.eval()

      all_true_rf, all_pred_rf = [], []
      all_true_trans, all_pred_trans = [], []

      with torch.no_grad():
        for images, labels_rf, labels_trans in test_loader:
          images = images.to(device)
          out_rf, out_trans = model(images)

          _, pred_rf = torch.max(out_rf, 1)
          _, pred_trans = torch.max(out_trans, 1)

          all_true_rf.extend(labels_rf.numpy())
          all_pred_rf.extend(pred_rf.cpu().numpy())
          all_true_trans.extend(labels_trans.numpy())
          all_pred_trans.extend(pred_trans.cpu().numpy())

      acc_rf = accuracy_score(all_true_rf, all_pred_rf)
      acc_trans = accuracy_score(all_true_trans, all_pred_trans)

      rf_accuracies.append(acc_rf)
      trans_accuracies.append(acc_trans)
    else:
      print(f"Warning: Missing model for Alpha {alpha}. Run training for Alpha {alpha} first.")
      rf_accuracies.append(None)
      trans_accuracies.append(None)

  plt.figure(figsize=(10,6))
  plt.plot(alphas, rf_accuracies, marker="o", linewidth=2, color="blue", label="Real/Fake Accuracy")
  plt.plot(alphas, trans_accuracies, marker="s", linewidth=2, color="green", label="Transformation Accuracy")

  plt.title("Ablation Study: Task Performance vs Loss Weight (Alpha)", fontsize=14)
  plt.xlabel("Alpha Value (Weight assigned for Real/Fake task)", fontsize=12)
  plt.ylabel("Test accuracy", fontsize=12)
  plt.xticks(alphas)
  plt.grid(True, linestyle="--", alpha=0.6)
  plt.legend(fontsize=11)

  plt.savefig(os.path.join(PLOTS_DIR, "ablation_study_summary.png"), dpi=300)
  plt.show()

# Data

In [ ]:
TRAIN_URL = "https://zenodo.org/records/14963880/files/RRDataset_original_train_val.tar.gz"

try:
  print("Downloading RRDataset_original_train_val.tar.gz...")
  exit_code = os.system(f"aria2c -x 16 -s 16 -d /content/ {TRAIN_URL}")

  if exit_code != 0:
    raise RuntimeError(f"aria2c terminal command failed with exit code {exit_code}")

  filepath = "/content/RRDataset_original_train_val.tar.gz"
  if not os.path.exists(filepath) or os.path.getsize(filepath) == 0:
    raise RuntimeError("File is missing or empty after download attempt")

  print("✓ Download completed successfully")
except Exception as e:
  print(f"✗ Error downloading RRDataset_original_train_val.tar.gz: {e}")
  raise

In [ ]:
TEST_URL = "https://zenodo.org/records/14963880/files/RRDataset_test.tar.gz"

try:
  print("Downloading RRDataset_test.tar.gz...")
  exit_code = os.system(f"aria2c -x 16 -s 16 -d /content/ {TEST_URL}")

  if exit_code != 0:
    raise RuntimeError(f"aria2c terminal command failed with exit code {exit_code}")

  filepath = "/content/RRDataset_test.tar.gz"
  if not os.path.exists(filepath) or os.path.getsize(filepath) == 0:
    raise RuntimeError("File is missing or empty after download attempt")

  print("✓ Download completed successfully")
except Exception as e:
  print(f"✗ Error downloading RRDataset_test.tar.gz: {e}")
  raise

In [ ]:
# Command line to unzip the archives
try:
    print("Extracting archives...")
    !tar -xzf /content/RRDataset_original_train_val.tar.gz
    !tar -xzf /content/RRDataset_test.tar.gz
    print("✓ Extraction completed successfully")

    # Remove the compressed archives to save memory
    !rm -rf /content/RRDataset_original_train_val.tar.gz
    !rm -rf /content/RRDataset_test.tar.gz
    print("Disk cleaned: .tar.gz files removed")

    if os.path.exists("RRDataset_original_train_val"):
      shutil.move("RRDataset_original_train_val", os.path.join(DATA_DIR, "RRDataset_original_train_val"))
    if os.path.exists("RRDataset_final"):
      shutil.move("RRDataset_final", os.path.join(DATA_DIR, "RRDataset_final"))
    print("✓ Files moved to the data/ directory")
except Exception as e:
    print(f"✗ Error extracting archives: {e}")
    raise

In [ ]:
#uniform the images for ViT, first one for training, second one for val and test
train_transform = transforms.Compose([
    #Instead of squashing the image, resize the short side to 256 pixels
    transforms.Resize(256),
    #Cut out a 224x224 square from the center
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    #The image is converted into a tensor so that PyTorch can process it
    transforms.ToTensor(),
    #The colors are normalized using ImageNet dataset mean and standard deviation
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

#there are no random flip and rotation cause at every test it should use the same
#pictures to ensure reproducibility
eval_transform = transforms.Compose([
    #Instead of squashing the image, resize the short side to 256 pixels
    transforms.Resize(256),
    #Cut out a 224x224 square from the center
    transforms.CenterCrop(224),
    #The image is converted into a tensor so that PyTorch can process it
    transforms.ToTensor(),
    #The colors are normalized using ImageNet dataset mean and standard deviation
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


class RRDataset(Dataset):
  def __init__(self, root_dirs_list, transform=None, samples_per_class=500):
    """takes an amount of samples from the dataset to define a class"""
    self.root_dirs = root_dirs_list
    self.transform = transform
    self.samples_per_class = samples_per_class

    #Lists that will be used by the DataLoader and during training
    self.image_paths = []   #List for the image paths
    self.labels_rf = []     #List for Real/Fake labels
    self.labels_trans = []  #List for the transformations labels

    #Dictionaries to translate words (classes) into labels
    #Some of the words are the official names of the classes, some of them are synonims
    #added so that samples with different labels can also be processed
    self.rf_map = {
        "real": 0, "nature": 0,
        "fake": 1, "ai": 1, "generated": 1, "synthetic": 1
    }
    self.trans_map = {
        "original": 0,
        "internet": 1, "transmitted": 1, "social":1, "transfer": 1,
        "redigitized": 2, "re-digitized": 2, "screen": 2, "redigital": 2
    }

    #Different combinations of classes
    #(0,0) = real and original, (1,0) = fake and original
    #(0,1) = real and transmitted, (1,1) = fake and transmitted
    #(0,2) = real and redigitized, (1,2) = fake and redigitized
    self.data_splits = {
        (0,0): [], (1,0): [],
        (0,1): [], (1,1): [],
        (0,2): [], (1,2): []
    }

    print("Scanning...")
    files = []
    for directory in self.root_dirs:
      #Selects dinamically all the files with .jpg, .png and .jpeg extension in any subdirectory
      files.extend(glob.glob(f"{directory}/**/*.jpg", recursive=True))
      files.extend(glob.glob(f"{directory}/**/*.png", recursive=True))
      files.extend(glob.glob(f"{directory}/**/*.jpeg", recursive=True))
    if not files:
      print("Error! No file found!\n")
      return;

    files.sort()

    #Iteration on every file path to initialize two variables that check if the
    #file can be processed
    for file_path in files:
      rf_label = -1
      trans_label = -1

      #Iteration on the words that indicate if the image is real or fake (the keys
      #of the rf_map dictionary)
      for key, val in self.rf_map.items():
        #If that word is present, the image will be labeled as indicated in the
        #dictionary (assigns 0 or 1)
        if key in file_path.lower():
          rf_label = val
          break

      #Iteration on the words that indicate if the image has been subjected to
      #transformations (keys of the trans_map dictionary)
      for key, val in self.trans_map.items():
        #If that word is present, the image will be labeled as indicated in the
        #dictionary (assigns 0, 1 or 2)
        if key in file_path.lower():
          trans_label = val
          break

      #If the labels changed, the image path is added into the corresponding list
      #of the data_split dictionary
      if rf_label != -1 and trans_label != -1:
        self.data_splits[(rf_label, trans_label)].append(file_path)

    #Iteration on every combination
    for key, paths_list in self.data_splits.items():
      #Decompose the tuple and take the corresponding elements
      rf_val, trans_val = key

      #How many samples to take with the current combination to avoid class to
      #be different in size
      num_samples = min(self.samples_per_class, len(paths_list))

      #Shuffles randomly the files and selects num_samples samples
      random.shuffle(paths_list)
      selected_paths = paths_list[:num_samples]

      #Iteration on the selected files
      for p in selected_paths:
        #Add the path
        self.image_paths.append(p)
        #Add the Real/Fake label
        self.labels_rf.append(rf_val)
        #Add the transformation label
        self.labels_trans.append(trans_val)

    print("Balanced Dataset Created")

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    try:
      #Get the path of the image in the idx position
      img_path = self.image_paths[idx]
      #Get both the corresponding labels (Real/Fake and Transformations)
      label_rf = self.labels_rf[idx]
      label_trans = self.labels_trans[idx]

      #The image is loaded in the RAM using the PIL library
      #With .convert("RGB") the image is converted into an RGB image (with 3 color channels)
      #so that it can be processed by PyTorch
      image = Image.open(img_path).convert("RGB")
      #If some transformations are meant to be applied to the image
      if self.transform:
        #Actually apply the transformations
        image = self.transform(image)

      return image, label_rf, label_trans
    except Exception as e:
      print(f"✗ Error loading image {img_path}: {e}")
      return None

In [ ]:
#IF YOU ARE TESTING A DIFFERENT VALUE OF ALPHA EXECUTE AGAIN THIS CELL
class DatasetWrapper(Dataset):
  """Wrapper to apply different transformations to different subsets"""
  def __init__(self, subset, transform=None):
    self.subset = subset
    self.transform = transform

  def __getitem__(self, index):
    """Exctract data from the original dataset"""
    image, label_rf, label_trans = self.subset[index]
    if self.transform:
      image = self.transform(image)
    return image, label_rf, label_trans

  def __len__(self):
    return len(self.subset)

try:
  #Create just one dataset
  #trasform= None so that train and test don't be subjected to same transformation
  full_dataset = RRDataset(
      root_dirs_list=[os.path.join(DATA_DIR, "RRDataset_final")],
      transform=None,
      samples_per_class=400
  )

  #70% Train, 15% Val, 15% Test
  total_size = len(full_dataset)
  train_size = int(0.7 * total_size)
  val_size = int(0.15 * total_size)
  test_size = total_size - train_size - val_size

  #Split the dataset according to the percentage defined above
  train_subset, val_subset, test_subset = random_split(
      full_dataset,
      [train_size, val_size, test_size],
      generator=torch.Generator().manual_seed(SEED)
  )

  #Apply transformations
  train_set = DatasetWrapper(train_subset, transform=train_transform)
  val_set = DatasetWrapper(val_subset, transform=eval_transform)
  test_set = DatasetWrapper(test_subset, transform=eval_transform)

  print(f"Train images:      {len(train_set)}")
  print(f"Validation images: {len(val_set)}")
  print(f"Test images:       {len(test_set)}")

  #Create DataLoaders
  #shuffle = True avoids change the order of input so that model doesn't overfit
  train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
  val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
  test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

  print("Configuration completed! Ready for training")

except Exception as e:
  print(f"Error during data preparation: {e}")
  raise

In [ ]:
class_balance(full_dataset, train_subset, val_subset, test_subset)

# Network

In [ ]:
class MultiTaskViT(nn.Module):
  def __init__(self):
    super(MultiTaskViT, self).__init__()

    # --- MAIN BODY (Shared backbone) ---
    #Import ViT-B/16 -> Vision Tranformer Base. With weights=...DEFAULT it's
    #specified that the imported architecture is pre-trained
    vit = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

    # Do not change the backbone weights, only the heads will be trained
    for param in vit.parameters():
      param.requires_grad = False

    #de-freeze only the last 3 layers. First layers elaborates general shapes
    #(lines,...), the last ones analize more abstract features
    for i in range(9, 12):
        for param in vit.encoder.layers[i].parameters():
            param.requires_grad = True
    vit.encoder.ln.requires_grad = True

    #After passing through all the Self-Attention layers, the image is mapped
    #into a vector of 768 elements (hidden state) that represents the conceptual
    #summary of the image
    self.num_features = 768

    #Replace ViT head with an identity layer(empty), so that data pass without being
    #converted into probabilities
    vit.heads = nn.Identity()
    #Save the main body without the head as vit
    self.backbone = vit

    # --- CLASSIFICATION HEADS ---
    #Real/fake section
    #Head 1 -> applies a linear projection: maps the num_features vector of 768
    #elements into a 2-element vector. The 2 raw numbers are the logits, and
    #represent how much the model thinks the image is Real or Fake. The logits
    #are not probabilities because they do not sum up to 1
    self.head_rf = nn.Sequential(
        nn.Linear(self.num_features, 256),     #from 768 to 256
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 2)                    #from 256 to 2
    )

    #transformation section
    #Head 2 -> maps the original vector into a 3-elements vector. Same concept
    #of the first head, but in this case the 3 logits represent how much the
    #model thinks the image is Original, Transmitted or Redigitalized
    self.head_trans = nn.Sequential(
        nn.Linear(self.num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 3)
    )

  #Defines the path that the data takes. PyTorch calls the function when doing
  #model(img)
  def forward(self, data):
    #Data enters in the backbone. Features is the hidden-state vector of 768 el.
    features = self.backbone(data)

    #Implementing MultiTasking by sending data to both the heads simultaneously
    #The first head calculates the Real/fake probabilities (saved in out_rf) the
    #second head calculates the Original/Transmitted/Redigitized probabilities
    #(saved in out_trans)
    out_rf = self.head_rf(features)
    out_trans = self.head_trans(features)

    return out_rf, out_trans

In [ ]:
#IF YOU ARE TESTING A DIFFERENT VALUE OF ALPHA EXECUTE AGAIN THIS CELL
#Model initialization -> create an instance of the model and moves the calculations
#and the weights on the GPU if available
model = MultiTaskViT().to(device)
print("Model created!")

#Loss function -> using 2 separate CrossEntropyLoss (one for each head) because
#it's a Multi Task environment that operates simultaneously in 2 different
#output spaces, one of dimension 2 (binary) and the other of dimension 3
loss_function_rf = nn.CrossEntropyLoss()
loss_function_trans = nn.CrossEntropyLoss()

#Using Adam optimizer that adapts automatically the learning rate to avoid
#stabilize the convergency. Share with adam only the unfreezed weights
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

print("Training is ready to start!")

# Train

In [ ]:
#IF YOU ARE TESTING A DIFFERENT VALUE OF ALPHA EXECUTE AGAIN THIS CELL
#Lists to memorize training history for the final plots
history = {
    "train_loss": [], "val_loss": [],
    "train_acc_rf": [], "val_acc_rf": [],
    "train_acc_trans": [], "val_acc_trans": []
}

# Initialize Early Stopping
early_stopping = EarlyStopping(patience=PATIENCE, verbose=True)

print(f"Starting training on {device} for {EPOCHS} epochs")

#Start training with catch of errors
try:
  for epoch in range(EPOCHS):
    #TRAINING PHASE
    model.train()
    train_loss = 0.0
    correct_rf = 0
    correct_trans = 0
    total = 0

    train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    #send images to device
    for images, labels_rf, labels_trans in train_loader_tqdm:
      images = images.to(device)
      labels_rf = labels_rf.to(device)
      labels_trans = labels_trans.to(device)

      #Reset gradients to avoid errors summation
      optimizer.zero_grad()

      #Forward pass -> the model returns the predictions for the real/fake and
      #the transformations labels
      pred_rf, pred_trans = model(images)

      #Calculate the two single losses and the total loss (multitasking)
      loss_rf = loss_function_rf(pred_rf, labels_rf)
      loss_trans = loss_function_trans(pred_trans, labels_trans)
      #ablation study giving different importance to each task
      total_loss = (ALPHA * loss_rf) + ((1- ALPHA) * loss_trans)

      #Backward pass and optimization to reduce errors
      total_loss.backward()
      optimizer.step()

      #Statistics
      train_loss += total_loss.item()

      #Real/Fake accuracy
      _, predicted_rf = torch.max(pred_rf.data, 1)
      correct_rf += (predicted_rf == labels_rf).sum().item()

      #Transformations accuracy
      _, predicted_trans = torch.max(pred_trans.data, 1)
      correct_trans += (predicted_trans == labels_trans).sum().item()

      total += labels_rf.size(0)

      #Update the writing on the progress bar
      train_loader_tqdm.set_postfix(loss=total_loss.item())

    #Calculate means of training epoch
    epoch_train_loss = train_loss/len(train_loader)
    epoch_train_acc_rf = correct_rf/total
    epoch_train_acc_trans = correct_trans/total

    #VALIDATION PHASE
    model.eval()
    val_loss = 0.0
    val_correct_rf = 0
    val_correct_trans = 0
    val_total = 0

    #Disable the gradient calculations to not waste memory
    #code is similar to training
    with torch.no_grad():
      for images, labels_rf, labels_trans in val_loader:
        images = images.to(device)
        labels_rf = labels_rf.to(device)
        labels_trans = labels_trans.to(device)

        pred_rf, pred_trans = model(images)

        loss_rf = loss_function_rf(pred_rf, labels_rf)
        loss_trans = loss_function_trans(pred_trans, labels_trans)

        val_loss += ((ALPHA * loss_rf) + (1 - ALPHA) * loss_trans).item()

        _, predicted_rf = torch.max(pred_rf.data, 1)
        val_correct_rf += (predicted_rf == labels_rf).sum().item()

        _, predicted_trans = torch.max(pred_trans.data, 1)
        val_correct_trans += (predicted_trans == labels_trans).sum().item()

        val_total += labels_rf.size(0)

    #Calculate mean validation epoch
    epoch_val_loss = val_loss/len(val_loader)
    epoch_val_acc_rf = val_correct_rf/val_total
    epoch_val_acc_trans = val_correct_trans/val_total

    #Saving history for plotting
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["train_acc_rf"].append(epoch_train_acc_rf)
    history["val_acc_rf"].append(epoch_val_acc_rf)
    history["train_acc_trans"].append(epoch_train_acc_trans)
    history["val_acc_trans"].append(epoch_val_acc_trans)

    #Print epoch results
    print(f"--- Epoch {epoch+1} results ---")
    print(f"Loss: Training {epoch_train_loss:.4f} | Validation {epoch_val_loss:.4f}")
    print(f"Real/Fake accuracy: Training {epoch_train_acc_rf:.4f} | Validation {epoch_val_acc_rf:.4f}")
    print(f"Transformations accuracy: Training {epoch_train_acc_trans:.4f} | Validation {epoch_val_acc_trans:.4f}")

    # Early stopping check
    early_stopping(epoch_val_loss)

    # Save checkpoint for best validation loss
    save_checkpoint(model, optimizer, epoch+1, epoch_val_loss)

    if early_stopping.counter == 0:
      save_best_model(model, optimizer, epoch+1, epoch_val_loss)

    if early_stopping.early_stop:
        print(f"✓ Early stopping triggered at epoch {epoch+1}")
        break

  print("✓ Training completed!")
  clean_checkpoints()

except Exception as e:
  print(f"✗ Error during training: {e}")
  raise

# Test

In [ ]:
def test(model, test_loader, device):
  model.eval()
  #dictionary to register labels
  results = {
      "true_rf": [], "predicted_rf": [],
      "true_trans": [], "predicted_trans": []
  }

  with torch.no_grad():
    for images, labels_rf, labels_trans in tqdm(test_loader, desc="Testing"):
      images = images.to(device)
      labels_rf = labels_rf.to(device)
      labels_trans = labels_trans.to(device)

      out_rf, out_trans = model(images)

      _, predicted_rf = torch.max(out_rf, 1)
      _, predicted_trans = torch.max(out_trans, 1)

      #update results with obtained values
      results["true_rf"].extend(labels_rf.cpu().numpy())
      results["predicted_rf"].extend(predicted_rf.cpu().numpy())
      results["true_trans"].extend(labels_trans.cpu().numpy())
      results["predicted_trans"].extend(predicted_trans.cpu().numpy())

  return pd.DataFrame(results)

In [ ]:
#IF YOU ARE TESTING A DIFFERENT VALUE OF ALPHA EXECUTE AGAIN THIS CELL
#Mapping setup
rf_map = {0: "Real", 1: "Fake"}
trans_map = {0: "Original", 1: "Internet", 2: "Re-digitized"}

#Weights loading and send to device
best_model = torch.load(os.path.join(MODELS_DIR, f"final_model_alpha{ALPHA}.pt"))
model.load_state_dict(best_model["model_state_dict"])
model.to(device)

#Start of test and print of results
test_results_df = test(model, test_loader, device)
performance_table = metrics(test_results_df, rf_map, trans_map)
plot_test_results(test_results_df, rf_map, trans_map)
print("\n")
ablation_plot()